# Week 8 Overview

This week will be a mix of data joining/merging problems and linear algebra. 
The first 5 problems are data cleaning and the final 4 problems are linear algebra. 

There are multiple ways to combine data. These methods are common cross multiple languages like pandas, SQL, and R. At times the naming is different but the general concepts apply. 

### **Joining or Merging**
This is a process of combining two datasets by adding the columns of one dataset to the other by some logical relationship between the columns. 

In SQL we call this joining but pandas has two functions:

**merge** - The default behavor for merge is to combine on columns matching.

**join** - The default behavor for join is to combine on the column index matching. 

Often times I will colloquially use the word "join" for either merging or joining in pandas. 

Left Dataset
| key    | value |
| -------- | ------- |
| A1  | $250    |
| A2 | $80     |
| A3    | $420    |

Right Dataset
| key    | different_value |
| -------- | ------- |
| A1  | cat    |
| A2 | dog     |
| A3    | apple    |

Data Joined on key

| key    | value | different_value |
| -------- | ------- | ------- |
| A1  | $250    | cat |
| A2 | $80     | dog |
| A3    | $420    | apple |

Typically we refer to the starting dataset as the left dataset and the one being added as the right. 

The logic is typically that there is the same value in a specific column in both datasets. SQL allows for slightly more advanced logic which we will learn next quarter. Today we will focus on just columns matching. 

There are different types of joins that you will explore in this notebook (inner, outer, left, right, cross). The typical visual that is used to illustration these concepts in Ven Diagrams. If you are getting stuck trying to pick the right join type search for "types of joins" and look at the pictures that come up.


## **Concat or Union**

This is a process of combining two dataset by adding the rows of one dataset to the end of another. There is no logic required for this. This is called conact in pandas and union in SQL. 

In most version of SQL you are required to have the same columns in both datasets. In pandas you don't have to. If I concatenate the two dataset above in pandas I would get:

| key    | value | different_value |
| -------- | ------- | ------- |
| A1  | $250    | null |
| A2 | $80     | null |
| A3    | $420    | null |
| A1  | null    | cat |
| A2 | null     | dog |
| A3    | null    | apple |

However if my right dataset looked like this:

| key    | value |
| -------- | ------- |
| A1  | cat    |
| A2 | dog     |
| A3    | apple    |

then I could union them in SQL or concat in pandas to get:

| key    | value |
| -------- | ------- |
| A1  | $250    |
| A2 | $80     |
| A3    | $420    |
| A1  | cat    |
| A2 | dog     |
| A3    | apple    |



In [2]:
import pandas as pd
import numpy as np

In [3]:
df_1 = pd.DataFrame({"ints": range(100)})
df_2 = pd.DataFrame({"ints": range(-10, 10)}, index=range(-10, 10))


df_1['threes'] = np.floor(df_1['ints']/3) * 3

df_2['evens'] = df_2['ints']*2
df_2['threes'] = np.floor(df_2['ints']/3) * 3

### Problem 1:

Your first task will be to create a dataset by `merging` `df_1` and `df_2` on the `ints` column where the match on both sides. The results will be a dataframe with 10 rows and 5 columns.

You will then create the same dataframe by using the `join` function and joining the two datasets where the indexes are equal. There will be a little more work of handle column duplication so look up the error and figure out arguments to set. How many columns do you get in this case?

In [13]:
m_merge = df_1.merge(df_2.reset_index(drop=True), on="ints", how="inner")
print(m_merge.shape)
m_merge.head() # merge on ints inner, threes is duplicated so it gets suffixes.
# 10 rows, 4 columns

(10, 5)


,ints,threes_x,evens,threes_y,threes_string
0,0,0.0,0,0.0,0.0
1,1,0.0,2,0.0,0.0
2,2,0.0,4,0.0,0.0
3,3,3.0,6,3.0,3.0
4,4,3.0,8,3.0,3.0


In [14]:
j = df_1.join(df_2, how="inner", lsuffix="_1", rsuffix="_2")
print(j.shape)
j.head() # join on matching indexes

# 10 rows, 5 columns

(10, 6)


,ints_1,threes_1,ints_2,evens,threes_2,threes_string
0,0,0.0,0,0,0.0,0.0
1,1,0.0,1,2,0.0,0.0
2,2,0.0,2,4,0.0,0.0
3,3,3.0,3,6,3.0,3.0
4,4,3.0,4,8,3.0,3.0


## Problem 2:

Next you will perform the same merge as above three times with the following modifications:

* You want to keep all rows in `df_1` even if there is no match found in `df_2`
* You want to keep all rows in `df_2` even if there is no match found in `df_1`
* You want to keep all rows in `df_1` and `df_2` even if there is no match found in the other dataframe


How many rows do you end up with in each case? 

Think through a scenario where you might want to do this and add it as a comment above each merge. 

In [6]:
# master list of something probably
left_keep  = df_1.merge(df_2, on="ints", how="left",  suffixes=("_1","_2"))
# lets you find stuff that isnt an error, like active employees and payroll
right_keep = df_1.merge(df_2, on="ints", how="right", suffixes=("_1","_2"))
# its a union so like making sure two databases are up to date or something
outer_keep = df_1.merge(df_2, on="ints", how="outer", suffixes=("_1","_2"))

print("left :", left_keep.shape[0])
print("right:", right_keep.shape[0])
print("outer:", outer_keep.shape[0])

left : 100
right: 20
outer: 110


### Problem 3

Now we are going to merge on columns that are not the same. Merge on the following:

* Merge `df_1` and `df_2` where `df_1.ints = df_2.evens`, only keep rows where there is a value for either dataframe
* Merge `df_1` and `df_2` where `df_1.ints = df_2.threes`, only keep rows where there is a value for either dataframe
* Merge `df_1` and `df_2` where `df_1.ints = df_2.threes`, keep all rows from `df_1` even if there is no match found in `df_2`
* Merge `df_1` and `df_2` where `df_1.threes = df_2.threes`, only keep rows where there is a value for either dataframe


How many rows do you end up with in each case? Are there any duplications? (try: value_count)

Think through a scenario where you might want to do this and add it as a comment above each merge. 

In [17]:
m1 = df_1.merge(df_2, left_on="ints", right_on="evens", how="inner")
print(m1.shape[0])

print(m1.value_counts())

10
ints_x  threes_x  ints_y  evens  threes_y  threes_string
0       0.0       0       0      0.0       0.0              1
2       0.0       1       2      0.0       0.0              1
4       3.0       2       4      0.0       0.0              1
6       6.0       3       6      3.0       3.0              1
8       6.0       4       8      3.0       3.0              1
10      9.0       5       10     3.0       3.0              1
12      12.0      6       12     6.0       6.0              1
14      12.0      7       14     6.0       6.0              1
16      15.0      8       16     6.0       6.0              1
18      18.0      9       18     9.0       9.0              1
Name: count, dtype: int64


In [22]:
m2 = df_1.merge(df_2, left_on="ints", right_on="threes", how="inner", suffixes=("_1","_2"))
print(m2.shape[0])

print(m2["ints_1"].value_counts())

10
ints_1
0    3
3    3
6    3
9    1
Name: count, dtype: int64


In [31]:
m3 = df_1.merge(df_2, left_on="ints", right_on="threes", how="left", suffixes=("_1","_2"))
print(m3.shape[0])

print(m3.value_counts())

106
ints_1  threes_1  ints_2  evens  threes_2  threes_string
0       0.0       0.0     0.0    0.0       0.0              1
                  1.0     2.0    0.0       0.0              1
                  2.0     4.0    0.0       0.0              1
3       3.0       3.0     6.0    3.0       3.0              1
                  4.0     8.0    3.0       3.0              1
                  5.0     10.0   3.0       3.0              1
6       6.0       6.0     12.0   6.0       6.0              1
                  7.0     14.0   6.0       6.0              1
                  8.0     16.0   6.0       6.0              1
9       9.0       9.0     18.0   9.0       9.0              1
Name: count, dtype: int64


In [26]:
m4 = df_1.merge(df_2, on="threes", how="inner", suffixes=("_1","_2"))
print(m4.shape[0])

# duplication check
print(m4["threes"].value_counts())

30
threes
0.0    9
3.0    9
6.0    9
9.0    3
Name: count, dtype: int64


### Problem 4

Add a new the column to `df_2` called `threes_string` that is the `threes` column converted to a string. Attempt to merge `df_1` and `df_2` where `df_1.threes = df_2.threes_string` with an inner join. What happens? Why?

In [25]:
df_2["threes_string"] = df_2["threes"].astype(str)

bad = df_1.merge(df_2, left_on="threes", right_on="threes_string", how="inner")

# i get a value error complaining about merging float 64 with object


ValueError: You are trying to merge on float64 and object columns for key 'threes'. If you wish to proceed you should use pd.concat

### Problem 5

Now you will play around with `pd.concat` by doing the following:

* Concatenate `df_1` and `df_2` keeping all rows, columns and indexes
* Concatenate `df_1` and `df_2` keeping all rows and columns but ignore the indexes from the orginal dataframes and instead have the index on this dataframe be zero to the number of rows.
* Concatenate `df_1` and `df_2` keeping all rows and indexes the same but only keeping columns that exist in both dataframes


In [ ]:
c1 = pd.concat([df_1, df_2], axis=0)
print(c1.shape)

(120, 4)


In [ ]:
c2 = pd.concat([df_1, df_2], ignore_index=True)
print(c2.shape, c2.index.min(), c2.index.max())

(120, 4) 0 119


In [ ]:
c3 = pd.concat([df_1, df_2], join="inner")
print(c3.shape, list(c3.columns))

(120, 2) ['ints', 'threes']


## Linear Algebra: Rank and Column Space

### Problem 6
You will now learn how to create random matrices with arbitrary rank (subject to the constraints about matrix sizes, etc.). To create an $m \times n$ matrix with rank $r$, multiply a random $m \times r$ matrix with a random $r \times n$ matrix. Implement this in Python and confirm that the rank is indeed $r$. 

What happens if you set $r > min{M,N}$, and why does that happen?

In [ ]:
def random_rank_mtx(m, n, r, seed=None):
    rng = np.random.default_rng(seed)
    A = rng.standard_normal((m, r))
    B = rng.standard_normal((r, n))
    return A @ B

for (m,n,r) in [(6,4,2),(5,7,3),(4,4,4)]:
    M = random_rank_mtx(m,n,r,seed=0)
    print((m,n,r), "rank =", np.linalg.matrix_rank(M))

# if r > min(m,n), The rank of an m×n matrix can’t exceed min(m,n), 
# so matrix_rank will cap out at min(m,n) even if you try to build with bigger r.

(6, 4, 2) rank = 2
(5, 7, 3) rank = 3
(4, 4, 4) rank = 4


### Problem 7
Interestingly, the matrices $A$, $A^T$, $A^T A$, and $AA^T$ all have the same rank. Write code to demonstrate this, using random matrices of various sizes, shapes (square, tall, wide), and ranks. Create a total of 6 random, two of each size that have different sizes and ranks. 

In [ ]:
def show_ranks(m, n, r, seed):
    A = random_rank_mtx(m,n,r,seed=seed)
    print(f"A {m}x{n} target r={r}:",
          np.linalg.matrix_rank(A),
          np.linalg.matrix_rank(A.T),
          np.linalg.matrix_rank(A.T @ A),
          np.linalg.matrix_rank(A @ A.T))

tests = [
    (6,6,2, 1), (6,6,4, 2),   # square (two different ranks)
    (8,3,2, 3), (8,3,3, 4),   # tall
    (3,9,2, 5), (3,9,3, 6),   # wide
]

for t in tests:
    show_ranks(*t)

A 6x6 target r=2: 2 2 2 2
A 6x6 target r=4: 4 4 4 4
A 8x3 target r=2: 2 2 2 2
A 8x3 target r=3: 3 3 3 3
A 3x9 target r=2: 2 2 2 2
A 3x9 target r=3: 3 3 3 3


### Problem 8

Demonstrate the addition rule of matrix rank $(r(A + B) ≤ r(A) + r(B))$ by creating three pairs of rank-1 matrices that have a sum with 
1. rank-0
2. rank-1
3. rank-2

Then repeat this exercise using matrix multiplication instead of addition.

In [ ]:
def r1(u, v):
    return np.outer(u, v)

rng = np.random.default_rng(0)
n = 4
u = rng.standard_normal(n)
v = rng.standard_normal(n)
A = r1(u, v)  # rank 1 (almost surely)

# rank-0 sum: B = -A
B0 = -A
S0 = A + B0

# rank-1 sum: B = cA with c != -1
c = 2.0
B1 = c*A
S1 = A + B1

# rank-2 sum: use independent outer products
u2 = rng.standard_normal(n)
v2 = rng.standard_normal(n)
A2 = r1(u, v)
B2 = r1(u2, v2)
S2 = A2 + B2

for name, M in [("S0",S0),("S1",S1),("S2",S2)]:
    print(name, np.linalg.matrix_rank(M))

S0 0
S1 1
S2 2


### Problem 9

The goal of this exercise is to answer the question is $v \in C(A)$?

Create a rank-3 matrix $A \in \mathbb{R}^{4 \times 3}$ and vector $v \in \mathbb{R}^{4}$ using numbers randomly drawn from a normal distribution. 

Follow the algorithm described in the [In the Column Space?](https://learning.oreilly.com/library/view/practical-linear-algebra/9781098120603/ch06.html#id335) section of Practical Linear Algebra to determine whether the vector is in the column space of the matrix. 

Rerun the code multiple times to see whether you find a consistent pattern. 

Next, use a $A \in \mathbb{R}^{4 \times 4}$ rank-4 matrix and a vector $v \in \mathbb{R}^{4}$ using numbers randomly drawn from a normal distribution. What happens in this case? Why?


In [ ]:
def in_col_space(A, v, tol=1e-10):
    Q, _ = np.linalg.qr(A)         # Q is m * m, but first r columns span Col(A) (numerically)
    r = np.linalg.matrix_rank(A)
    Qr = Q[:, :r]
    v_proj = Qr @ (Qr.T @ v)
    resid = np.linalg.norm(v - v_proj)
    return resid, resid < tol

rng = np.random.default_rng(0)

# A is 4x3 rank 3, v in R^4
A = random_rank_mtx(4,3,3,seed=1)   # rank 3 maybe
v = rng.standard_normal(4)

resid, inside = in_col_space(A, v)
print("4x3 rank-3 residual:", resid, "in C(A)?", inside)

4x3 rank-3 residual: 0.5865398963550247 in C(A)? False


You will almost always get “False” because the column space of A is only a 3-dimensional slice inside 4-dimensional space, and a randomly chosen vector in 4D space is extremely unlikely to lie exactly in that slice

In [ ]:
A = random_rank_mtx(4,4,4,seed=2)   # full rank 4 (almost surely)
v = rng.standard_normal(4)

resid, inside = in_col_space(A, v)
print("4x4 rank-4 residual:", resid, "in C(A)?", inside)

4x4 rank-4 residual: 2.2887833992611187e-16 in C(A)? True
